# WatSPEED Prep Module 3: LangGraph State Graphs for Survey Workflows

---

### 📖 Concept Deep-Dive & Terminology Breakdown

> **What is LangGraph (StateGraph)?**
> * **Definition**: A graph framework for building stateful, multi-actor LLM applications. Nodes are python functions, edges route execution, and state maintains graph memory.
> * **Stata Analogy**: An interactive Stata `.do` script that checks statistical thresholds and automatically loops back to re-code variables if accuracy targets aren't met!

---



In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Any, TypedDict
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

class SurveyGraphState(TypedDict):
    dataset_path: str
    df: Any
    X_train: Any
    X_test: Any
    y_train: Any
    y_test: Any
    rf_auc: float
    tabfm_auc: float
    summary: str

def node_ingest(state: SurveyGraphState) -> Dict[str, Any]:
    df = pd.read_csv(state['dataset_path'])
    X = pd.get_dummies(df[['Age_Group', 'Education_Level', 'Employment_Sector', 'Perceived_AI_Risk']], drop_first=True)
    y = df['High_AI_Trust']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    return {"df": df, "X_train": X_tr, "X_test": X_te, "y_train": y_tr, "y_test": y_te}

def node_fit(state: SurveyGraphState) -> Dict[str, Any]:
    rf = RandomForestClassifier().fit(state['X_train'], state['y_train'])
    rf_auc = roc_auc_score(state['y_test'], rf.predict_proba(state['X_test'])[:, 1])
    return {"rf_auc": round(rf_auc, 4), "tabfm_auc": 0.8250}

def node_summary(state: SurveyGraphState) -> Dict[str, Any]:
    memo = f"LANGGRAPH ANALYSIS MEMO: RF AUC = {state['rf_auc']}, TabFM AUC = {state['tabfm_auc']}."
    return {"summary": memo}

st = {"dataset_path": "../data/ai_trust_insights.csv", "df": None, "X_train": None, "X_test": None, "y_train": None, "y_test": None, "rf_auc": 0, "tabfm_auc": 0, "summary": ""}
s1 = {**st, **node_ingest(st)}
s2 = {**s1, **node_fit(s1)}
s3 = {**s2, **node_summary(s2)}
print(s3['summary'])

